# 09 | Métricas y comparación

> **Cuaderno de reproducción (posterior).** Generado el 2026-10-23 como parte de la entrega docente. El repositorio original no contenía cuadernos versionados; estos reconstruyen los análisis a partir de los scripts y módulos existentes.

**Tipo de ejecución: RECÁLCULO — calcula métricas desde predicciones guardadas (y_true/y_pred); NO entrena.**

In [1]:
import os
from pathlib import Path

def _raiz_repositorio() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'data' / 'raw' / 'interseccion_terrenos_zhg_r1_yacopi.csv').exists():
            return p
    raise FileNotFoundError('No se encontro la raiz del repositorio (busca data/raw)')

REPO = _raiz_repositorio()
print('Raiz del repositorio resuelta (carpeta):', REPO.name, '| data/raw encontrado')

Raiz del repositorio resuelta (carpeta): yacopi-project | data/raw encontrado


## Recálculo desde pares y_true/y_pred

Se leen las predicciones guardadas por `audit_r2.py` (protocolo v2) y se recalculan MAE, RMSE y R² con sklearn para los 5 modelos del escenario A en ambos protocolos, contrastando con `resumen.json`.

In [2]:
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np, json
modelos = ['dummy_media','arbol','random_forest','ridge_log','catboost']
base = REPO / 'ml/artifacts/exp-valor-terreno-001/audit_r2'
resumen = json.loads((REPO / 'ml/artifacts/exp-valor-terreno-001/resumen.json').read_text(encoding='utf-8'))
filas = []
for particion in ['test_predio','test_espacial']:
    for modelo in modelos:
        d = pd.read_csv(base / f'preds_A_{modelo}_{particion}.csv')
        y, yp = d.y_true.to_numpy(), d.y_pred.to_numpy()
        m = {'modelo': modelo, 'particion': particion, 'n': len(d),
             'mae': mean_absolute_error(y, yp),
             'rmse': float(np.sqrt(mean_squared_error(y, yp))),
             'r2': r2_score(y, yp)}
        filas.append(m)
tabla = pd.DataFrame(filas)
print(tabla.round(3).to_string(index=False))

       modelo     particion   n         mae        rmse     r2
  dummy_media   test_predio 936  758752.256 1993974.228 -0.000
        arbol   test_predio 936  343025.398 1466263.943  0.459
random_forest   test_predio 936  248296.243 1111206.802  0.689
    ridge_log   test_predio 936  392022.181 1665458.041  0.302
     catboost   test_predio 936  279283.899 1044694.172  0.725
  dummy_media test_espacial 689 1237010.645 3800734.508 -0.006
        arbol test_espacial 689  805689.985 3610639.595  0.092
random_forest test_espacial 689  767064.076 3450750.628  0.171
    ridge_log test_espacial 689  838995.808 3769143.439  0.011
     catboost test_espacial 689  897642.467 3929401.584 -0.075


In [3]:
print('Contraste con resumen.json (deben coincidir):')
for _, f in tabla.iterrows():
    r = resumen['A']['protocolos'][{'test_predio':'predio','test_espacial':'espacial'}[f.particion]]['resultados'][f.modelo]
    ok = (abs(r['mae_cop_ha'] - f.mae) < 1) and (abs(r['r2'] - f.r2) < 1e-6)
    print(f"  {f.modelo:14s} {f.particion:14s} coincide={ok}")

Contraste con resumen.json (deben coincidir):
  dummy_media    test_predio    coincide=True
  arbol          test_predio    coincide=True
  random_forest  test_predio    coincide=True
  ridge_log      test_predio    coincide=True
  catboost       test_predio    coincide=True
  dummy_media    test_espacial  coincide=True
  arbol          test_espacial  coincide=True
  random_forest  test_espacial  coincide=True
  ridge_log      test_espacial  coincide=True
  catboost       test_espacial  coincide=True


## Lectura

Las métricas recalculadas aquí provienen de predicciones generadas previamente por `audit_r2.py`; este cuaderno NO reentrena.